# 06 Feedback loop: matched ratio at increasing scale

Tests the production **mix**: every student is trained with derived rows / organic rows equal to the
production ratio. Because the experiment needs held-out folds, it can only do that with fewer organic
rows than production, and with a teacher trained on fewer rows (so noisier pseudo-labels). So the
matched-ratio comparison is repeated at several scales. If the gain is stable as the scale grows, the
remaining step to full production size is small and safe to extrapolate; if it trends, the trend
line estimates the production value.

Each scale *s*, per test fold *k*:

1. organic rows: a nested hash subsample (share *s*) of the organic rows outside fold *k*
2. teacher: trained on those rows, as production trains on all organic rows (the "without" model)
3. the teacher re-labels the pool and the acceptance rule is re-applied; production labels are never used
4. student ("with"): the same organic rows plus `R_PROD x organic` accepted pool items
5. both are scored on fold *k*

Checkpointed in `outputs/feedback_matched_ratio_<timestamp>/`; set `RESUME_FROM` to continue.

In [ ]:
# ---- MLflow -------------------------------------------------------------------
USE_MLFLOW = False                    # False = no tracking at all
MLFLOW_EXPERIMENT = None              # None = this notebook's experiment; or e.g. "/Users/patrick.white@vallen.com/nb-classifier"

# ---- experiment -----------------------------------------------------------------
N_FOLDS = 3                           # ID-hash folds of the organic rows (test folds)
FOLD_SEED = 0
FOLDS_TO_RUN = None                   # None = all; e.g. [0] for a quick first look
SCALES = [0.1, 0.2, 0.35, 0.5, 0.7, 1.0]  # shares of the training folds' organic rows (nested subsamples)
SAMPLE_SEED = 0                       # seed of the nested subsamples
PRODUCTION_RATIO = None               # None = derived rows / organic rows in the data
DEPTH = None                          # level the analysis uses (None = deepest, i.e. the full path)
N_BOOT = 300                          # paired bootstrap replicates over test rows
CI = 0.90                             # width of the intervals
RATIO_TOLERANCE = 0.05                # flag scales whose realized ratio is more than 5% below the target
TREND_MIN_SCALE = 0.35                # the trend is fitted on scales >= this; the smallest scales are in a
                                      # small-data regime and would dominate a straight line in log(n)
RESUME_FROM = None                    # e.g. "feedback_matched_ratio_20261020_143000"

In [ ]:
# ---- prepared data ------------------------------------------------------------
PREPARED_DATA = None                  # a data/<timestamp> folder from notebook 01 ("latest" = newest);
                                      # None = pull and clean below with the settings in this cell

# ---- data source (RAW_DATA_FILE supersedes SQL_QUERY) -----------------------
RAW_DATA_FILE = None                  # e.g. "data/my_extract.txt"; read with dtype=str
RAW_DATA_DELIMITER = "|"              # delimiter of RAW_DATA_FILE
RAW_DATA_QUOTED = True                # False if RAW_DATA_FILE uses no quoting and descriptions may start with "
SQL_QUERY = "labeled_items"           # a file in data/queries/ (name without .sql); may be None only if RAW_DATA_FILE is set

# ---- items to exclude ---------------------------------------------------------
BLACKLIST_FILE = None                 # e.g. "data/blacklists/blacklist.txt" (one column)
BLACKLIST_COL = "Vallen ID"           # data column the blacklist values are matched against
BLACKLIST_HAS_HEADER = True           # first line of the blacklist file is a header

# ---- cleaning -----------------------------------------------------------------
CLEANING_METHOD = "cleaning_method_20261015"   # a folder in cleaning/ (code, term lists, settings in method.yaml)
APPEND_FIELDS = {}                    # e.g. {"Global Vendno": "VEND", "Prodline": "PLINE"} -> VEND_<value> tokens
APPEND_NULLS_AS_NONE = True           # True: null field -> VEND_None;  False: no marker for that row

# ---- columns ------------------------------------------------------------------
TEXT_COL = "FullDesc"                 # raw description field to clean
CLEANED_COL = "CleanDesc"             # header for the cleaned description
LEVEL_COLUMNS = ["Level 1 ID", "Level 2 ID", "Level 3 ID", "Level 4 ID"]      # labels the model learns
ID_COL = "ID"                         # item key: matched against the prediction files, hashed for folds and subsamples

# ---- which labelled rows came from predictions ----------------------------------
PREDICTION_FILES = "all"              # notebook-04 outputs the derived labels came from: "all" (every
                                      # outputs/predictions_*), a folder name, a CSV path, or a list of these.
                                      # A labelled row whose ID is in these files counts as derived.
DERIVED_FLAG_COL = None               # or a data column marking derived rows (1/true/yes); overrides the ID match
POOL_SOURCE = "prediction_files"      # "prediction_files": every predicted item (accepted or not) is re-labelled
                                      #   by the fold teachers (needs INCLUDE_TEXT = True in notebook 04);
                                      # "derived": only the derived rows
ACCEPT_LEVEL = 4                      # predictions were accepted on this level's confidence ...
ACCEPT_MIN_CONFIDENCE = None          # ... when it was at least this; None = every prediction was accepted.
                                      # The diagnostics table below shows the threshold that was actually in use.

In [ ]:
# ==== HYPERPARAMETERS ==========================================================
# Single values only: the production model's settings (as in notebook 03). Every teacher and
# student in this notebook is trained with them.

HYPERPARAMETERS = {
    "vectorizer_kind":     "tfidf",
    "max_features":        None,
    "ngram_range":         (1, 2),
    "min_df":              2,
    "max_df":              0.95,
    "stop_words":          "english",
    "sublinear_tf":        True,
    "binary":              True,
    "norm":                "l2",
    "use_idf":             False,
    "ordered_bigrams":     True,
    "token_pattern":       "whitespace",
    "alpha":               0.001,
    "fit_prior":           True,
    "prediction_mode":     "conditional",
    "rare_path_threshold": 5,
}

In [ ]:
# ---- setup (no settings below this line need changing) ----------------------
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from nbclassifier import config, data, feedback, paths
from nbclassifier.tracking import Tracker

KIND = "matched_ratio"
hp = config.expand_grid(HYPERPARAMETERS, grid_search=False)[0]
rule = feedback.SelectionRule(level=ACCEPT_LEVEL, min_confidence=ACCEPT_MIN_CONFIDENCE)
CONFIG = {
    "hyperparameters": hp.to_dict(), "n_folds": N_FOLDS, "fold_seed": FOLD_SEED, "sample_seed": SAMPLE_SEED,
    "rule": rule.to_dict(),
}
tracker = Tracker(USE_MLFLOW, MLFLOW_EXPERIMENT, run_name=f"06_feedback_matched_ratio_{paths.timestamp()}",
                  tags={"notebook": "06_feedback_matched_ratio"})

## Data

Organic rows are split from derived rows, and the pool (everything the model predicted) is built.
Check the diagnostics table:

* **recorded label = production prediction**: 1.0 means the derived labels are pure model output.
  Less than 1.0 means some were corrected after prediction, so they are partly validated. The
  experiments never use the recorded labels: the fold teachers re-label the pool, because the production
  model has seen every test fold. That is exact when the recorded labels are the model's predictions;
  the more were corrected, the more these experiments understate the real derived labels' quality.
* **production confidence of accepted vs not-accepted pool items**: if the two ranges don't
  overlap, the minimum of the accepted ones is the acceptance threshold that was in use. Set
  `ACCEPT_MIN_CONFIDENCE` to it so the fold teachers accept items the same way.

In [ ]:
if RESUME_FROM:
    store = feedback.Store.open(RESUME_FROM, KIND, CONFIG)
    fd = store.data()
    cols = feedback.Cols(**store.config["columns"])
    print(f"Resuming {store.folder.name}: the data saved in that folder is used; data settings above are ignored.")
else:
    if PREPARED_DATA:
        prepared = data.load_dataset(PREPARED_DATA)
        print(f"loaded {prepared.metadata['loaded_from']} (cleaning method {prepared.cleaner.method_name!r})")
        prepared = data.blacklist_prepared(prepared, BLACKLIST_FILE, BLACKLIST_COL, BLACKLIST_HAS_HEADER)
        TEXT_COL, CLEANED_COL = prepared.metadata["text_col"], prepared.metadata["cleaned_col"]
    else:
        prepared = data.prepare_dataset(
            raw_data_file=RAW_DATA_FILE, sql_query=SQL_QUERY, delimiter=RAW_DATA_DELIMITER,
            blacklist_file=BLACKLIST_FILE, blacklist_col=BLACKLIST_COL, blacklist_has_header=BLACKLIST_HAS_HEADER,
            cleaning_method=CLEANING_METHOD,
            append_fields=APPEND_FIELDS, append_nulls_as_none=APPEND_NULLS_AS_NONE,
            text_col=TEXT_COL, cleaned_col=CLEANED_COL, level_columns=LEVEL_COLUMNS,
            raw_data_quoted=RAW_DATA_QUOTED, spark=globals().get("spark"),
        )
    fd = feedback.split_feedback(prepared, level_columns=LEVEL_COLUMNS, text_col=TEXT_COL, cleaned_col=CLEANED_COL,
                                 id_col=ID_COL, prediction_files=PREDICTION_FILES,
                                 derived_flag_col=DERIVED_FLAG_COL, pool_source=POOL_SOURCE)
    del prepared
    cols = feedback.Cols(id=ID_COL, cleaned=CLEANED_COL, levels=list(LEVEL_COLUMNS))
    store = feedback.Store.create(KIND, {**CONFIG, "columns": cols.__dict__}, fd)
LEVEL_COLUMNS = cols.levels
DEPTH = DEPTH or len(LEVEL_COLUMNS)
N_PROD = len(fd.organic)                       # organic rows the production model is trained on
R_PROD = PRODUCTION_RATIO if PRODUCTION_RATIO is not None else fd.production_ratio
print(f"\nproduction: {N_PROD:,} organic rows, ratio {R_PROD:.3f}; analysis at L{DEPTH}")
print("evaluation folder:", store.folder)
feedback.describe(fd, LEVEL_COLUMNS, rule)

## Run

Two models per (fold, scale): the teacher ("without") and the student at the production ratio ("with").

In [ ]:
R_TARGET = round(float(R_PROD), 4)
results = feedback.run_ratio_grid(
    store, fd, hp, cols, n_folds=N_FOLDS, fold_seed=FOLD_SEED, folds_to_run=FOLDS_TO_RUN,
    organic_fracs=SCALES, ratios=[R_TARGET], rule=rule, sample_seed=SAMPLE_SEED,
)

## With vs without at each scale

`teacher error among accepted` is the error rate of the teacher's predictions that pass the acceptance
rule, measured on the clean test fold. It is the closest observable proxy for the noise in the
pseudo-labels at that scale, and it should fall as the scale grows.

In [ ]:
meta = store.meta()
meta = meta[meta["organic_frac"].isin(SCALES) & meta["ratio_target"].isin([0.0, R_TARGET])]
meta["cell"] = list(zip(meta["organic_frac"], meta["ratio_target"]))
n_cells = meta["cell"].nunique()
complete = meta.groupby("fold")["cell"].nunique()
folds_done = sorted(complete[complete == n_cells].index)
if not folds_done:
    raise RuntimeError("No fold has finished every scale yet")
meta = meta[meta["fold"].isin(folds_done)]
print(f"folds used: {folds_done}")

scales = sorted(meta["organic_frac"].unique())
cells = [(s, 0.0) for s in scales] + [(s, R_TARGET) for s in scales]
keys = [{int(r.fold): r.key for r in meta[meta["cell"] == c].itertuples()} for c in cells]
point, boot = feedback.bootstrap_cells(store, keys, DEPTH, n_boot=N_BOOT)
S = len(scales)
d_point, d_boot = point[S:] - point[:S], boot[:, S:] - boot[:, :S]
q = [(1 - CI) / 2, 1 - (1 - CI) / 2]

w_meta = meta[meta["ratio_target"] == R_TARGET].groupby("organic_frac")
o_meta = meta[meta["ratio_target"] == 0.0].groupby("organic_frac")
table = pd.DataFrame({
    "scale": scales,
    "n_organic": o_meta["n_organic"].mean().loc[scales].round().astype(int).to_numpy(),
    "n_derived": w_meta["n_derived"].mean().loc[scales].round().astype(int).to_numpy(),
    "ratio_realized": w_meta["ratio"].mean().loc[scales].to_numpy(),
    "teacher error among accepted": o_meta[f"teacher L{DEPTH} error among accepted"].mean().loc[scales].to_numpy(),
    "accuracy without": point[:S], "accuracy with": point[S:],
    "delta": d_point, "delta_low": np.quantile(d_boot, q[0], axis=0), "delta_high": np.quantile(d_boot, q[1], axis=0),
    "delta_se": d_boot.std(axis=0, ddof=1),
})
short = table["ratio_realized"] < R_TARGET * (1 - RATIO_TOLERANCE)
if short.any():
    print(f"WARNING: at scale(s) {table.loc[short, 'scale'].tolist()} the teacher accepted too few pool items "
          f"to reach ratio {R_TARGET}; those points test a lower ratio and are left out of the trend fit.")
table.round(4)

## Trend across scales and extrapolation to production size

`delta = a + b*log(n_organic)` is fitted on the scales from `TREND_MIN_SCALE` up (weighted, and refitted
on every bootstrap replicate). The slope says whether the gain is stable; the line at `N_PROD` estimates
the production gain. The step from the largest tested scale to production is only
`N_FOLDS / (N_FOLDS - 1)` in organic rows, so this extrapolation is short. Hollow points in the plot
are not used in the fit.

In [ ]:
ok = ~short.to_numpy() & (table["scale"] >= TREND_MIN_SCALE).to_numpy()
if ok.sum() < 2:
    raise RuntimeError("Fewer than two fitted scales reached the target ratio; add larger scales, lower "
                       "TREND_MIN_SCALE or check ACCEPT_MIN_CONFIDENCE")
n = table.loc[ok, "n_organic"].to_numpy()
w = 1.0 / np.maximum(table.loc[ok, "delta_se"].to_numpy(), 1e-6) ** 2
coef = feedback.fit_log_trend(n, table.loc[ok, "delta"].to_numpy(), w).ravel()
coef_boot = feedback.fit_log_trend(n, d_boot[:, ok].T, w)          # 2 x N_BOOT
slope_lo, slope_hi = np.quantile(coef_boot[1], q)
at_prod = coef[0] + coef[1] * np.log(N_PROD)
at_prod_boot = coef_boot[0] + coef_boot[1] * np.log(N_PROD)
lo, hi = np.quantile(at_prod_boot, q)
per_double = coef[1] * np.log(2)

print(f"slope: {per_double:+.4f} accuracy per doubling of organic rows "
      f"({CI:.0%} interval {slope_lo * np.log(2):+.4f} to {slope_hi * np.log(2):+.4f})")
if slope_lo <= 0 <= slope_hi:
    print("  => no detectable trend: the matched-ratio gain is stable across the scales tested")
else:
    print(f"  => the gain {'grows' if coef[1] > 0 else 'shrinks'} with scale; use the extrapolated value")
print(f"\nat production size ({N_PROD:,} organic rows, ratio {R_PROD:.3f}): "
      f"delta {at_prod:+.4f}  ({CI:.0%} interval {lo:+.4f} to {hi:+.4f})")
largest = table.loc[ok].iloc[-1]
print(f"at the largest tested scale ({int(largest['n_organic']):,} rows): delta {largest['delta']:+.4f} "
      f"({largest['delta_low']:+.4f} to {largest['delta_high']:+.4f})")
if hi < 0:
    print("=> including the derived items is expected to LOWER accuracy")
elif lo > 0:
    print("=> including the derived items is expected to RAISE accuracy")
else:
    print("=> the interval includes zero: no reliable gain or loss")
tracker.log_metrics({"delta_at_production": at_prod, "delta_low": lo, "delta_high": hi,
                     "slope_per_doubling": per_double, "production_ratio": R_PROD})

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(table["n_organic"], table["accuracy without"], "o-", color="steelblue", label="without (teacher)")
ax1.plot(table["n_organic"], table["accuracy with"], "o-", color="firebrick", label=f"with, ratio {R_TARGET}")
ax1.set_xscale("log")
ax1.set_xlabel("organic training rows")
ax1.set_ylabel(f"L{DEPTH} accuracy on clean test folds")
ax1.set_title("Accuracy at the production mix, by scale")
ax1.legend()

for used, style in ((True, dict(fmt="o", color="black", label="measured (fitted)")),
                    (False, dict(fmt="o", color="black", mfc="white", label="measured (not fitted)"))):
    t = table[ok == used]
    if len(t):
        ax2.errorbar(t["n_organic"], t["delta"], yerr=[t["delta"] - t["delta_low"], t["delta_high"] - t["delta"]],
                     capsize=3, **style)
if short.any():
    ax2.scatter(table.loc[short, "n_organic"], table.loc[short, "delta"], marker="x", color="orange", s=60,
                label="ratio not reached (not fitted)", zorder=5)
nn = np.geomspace(table.loc[ok, "n_organic"].min() * 0.9, N_PROD * 1.1, 100)
band = coef_boot[0][:, None] + coef_boot[1][:, None] * np.log(nn)[None, :]
ax2.plot(nn, coef[0] + coef[1] * np.log(nn), color="firebrick", lw=1.2, label="trend")
ax2.fill_between(nn, np.quantile(band, q[0], axis=0), np.quantile(band, q[1], axis=0), color="firebrick", alpha=0.15)
ax2.axvline(N_PROD, color="gray", ls=":", lw=1)
ax2.errorbar([N_PROD], [at_prod], yerr=[[at_prod - lo], [hi - at_prod]], fmt="D", color="firebrick", capsize=4,
             label=f"production size: {at_prod:+.4f}")
ax2.axhline(0, color="gray", lw=0.8)
ax2.set_xscale("log")
ax2.set_xlabel("organic training rows")
ax2.set_ylabel(f"change in L{DEPTH} accuracy (with - without)")
ax2.set_title(f"Gain at ratio {R_TARGET} by scale")
ax2.legend(fontsize=8)
fig.tight_layout()
plot_dir = store.folder / "plots"
plot_dir.mkdir(exist_ok=True)
fig.savefig(plot_dir / f"matched_ratio_L{DEPTH}.png", dpi=120, bbox_inches="tight")
tracker.log_figure(fig, f"plots/matched_ratio_L{DEPTH}.png")

## By level, at the largest scale

In [ ]:
s_max = table.loc[ok, "scale"].max()
pair_keys = [{int(r.fold): r.key for r in meta[meta["cell"] == c].itertuples()} for c in [(s_max, 0.0), (s_max, R_TARGET)]]
rows = []
for d in range(1, len(LEVEL_COLUMNS) + 1):
    p, b = feedback.bootstrap_cells(store, pair_keys, d, n_boot=N_BOOT, seed=d)
    db = b[:, 1] - b[:, 0]
    rows.append({"level": f"L{d}", "accuracy without": p[0], "accuracy with": p[1], "delta": p[1] - p[0],
                 "delta_low": np.quantile(db, q[0]), "delta_high": np.quantile(db, q[1])})
pd.DataFrame(rows).set_index("level").round(4)

In [ ]:
table.to_csv(store.folder / f"scales_L{DEPTH}.csv", index=False)
tracker.log_table(table, f"scales_L{DEPTH}.csv")
tracker.end()